# Find Optimal Solutions

**Where this fits:** `04_optimize.ipynb` produced 1,000 candidate allocations
by searching the *metamodel's* predicted landscape. But the metamodel is a
learned approximation, not the real thing — it's not guaranteed to be exactly
right everywhere. Before trusting its top picks, we go back to the actual
simulation one more time and check: for the handful of allocations the
metamodel likes best, does the real simulation agree? That validation step is
what makes the final "342.26 – 343.03 projected deaths" figure in the README
a *simulated*, ground-truth number — not just a model's guess. The output
here (`metamodel_results.csv`) is one of the three inputs `07_compare_results.ipynb`
uses for the final head-to-head comparison, alongside the status quo (`00`)
and greedy (`06`) results.

**Input:** `results/outputs/Local_Minima.csv` — the 1,000 local minima produced in `04_optimize.ipynb`

**Output:** `results/outputs/metamodel_results.csv` — the best candidate allocations, re-evaluated through the actual simulation

**Workflow:**

1. Select the best candidate local minima found by the metamodel.
2. Re-run those candidate allocations through the actual simulation (on the HPC cluster) to validate the metamodel's predictions against ground truth.
3. Combine and summarize the re-simulated results for comparison against the other approaches.</cell id="c029004e">

1. Start by importing the file that contains all the local minima. 

In [ ]:
from pathlib import Path

# Current notebook directory
cwd = Path.cwd()

# Navigate to repo root (go up 1 level)
repo_root = cwd.parents[0]

# Build file path
file_path = repo_root / "results" / "outputs" / "Local_Minima.csv"

file_path

In [ ]:
import pandas as pd 

final_df = pd.read_csv(file_path)
print(final_df.shape)

2. Plot the distribution to get a visual of the different local minima

In [ ]:
import matplotlib.pyplot as plt

# Create histogram
final_df['Minimized_Value'].hist(bins=20, edgecolor='black')  # Adjust `bins` as needed
plt.xlabel('Projected 2026 Deaths')
plt.ylabel('Frequency')
plt.title('Histogram of Local Minima results')
plt.show()

3. Find the 100 rows with the smallest local minima. 

In [ ]:
# Get the 100 rows with the smallest values and reset index
smallest_rows = (
    final_df
    .nsmallest(100, 'Minimized_Value')
    .reset_index(drop=True)
)

print("Top 100 rows with the smallest Minimized_Value:")
print(smallest_rows)

In [ ]:
smallest_rows = smallest_rows.iloc[:, 1:]
print(smallest_rows.shape)

In [ ]:
from pathlib import Path

# Current notebook directory
cwd = Path.cwd()

# Navigate to repo root
repo_root = cwd.parents[0]

# Folder to save outputs
output_dir = repo_root / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the top 100 candidate allocations (pre re-simulation)
df_path = output_dir / "candidate_solutions.csv"
smallest_rows.to_csv(df_path, index=True)

print(f"DataFrame saved to: {df_path}")

We now take the top candidate allocations and prepare them to be re-run through
the actual simulation, so that the metamodel's predicted rankings can be checked
against ground-truth simulated outcomes. Each candidate allocation is duplicated
across the 50 calibrated parameter seeds used elsewhere in this project, so that
each candidate gets a stable, averaged death estimate rather than a single noisy
simulation run.

In [ ]:
import pandas as pd

# Initialize an empty list to hold duplicated rows
duplicated_rows = []

# Process the first 16 rows
for i in range(16):
    # Step 1: Extract the row as a DataFrame
    row_df = smallest_rows.iloc[[i]].copy()
    
    # Step 2: Remove the first and last columns
    trimmed_row = row_df.iloc[:, 1:-1]
    
    # Step 3: Replace spaces in column names with '.'
    trimmed_row.columns = trimmed_row.columns.str.replace(' ', '.', regex=False)
    
    # Step 4: Duplicate the row 50 times
    duplicated = pd.concat([trimmed_row] * 50, ignore_index=True)
    
    # Append to the list
    duplicated_rows.append(duplicated)

# Concatenate all duplicated rows into one DataFrame
final_df = pd.concat(duplicated_rows, ignore_index=True)

print(final_df)
print(f"Final shape: {final_df.shape}")  # Should be (800, number of columns after trimming)

In [ ]:
# Current notebook directory
cwd = Path.cwd()

# Navigate to repo root
repo_root = cwd.parents[0]

# Folder to save outputs
output_dir = repo_root / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the candidate allocations duplicated across parameter seeds, ready for
# re-simulation on the HPC cluster
file_path = output_dir / "candidate_solutions_for_resimulation.csv"
final_df.to_csv(file_path, index=False)

print(f"Saved to: {file_path}")

> **Note:** `candidate_solutions_for_resimulation.csv` was run back through the
> actual simulation on the HPC cluster over the 50 calibrated parameter seeds.
> The resulting per-seed outputs were then combined and averaged in the next
> section to produce a single, stable death estimate per candidate allocation.

## Combine Re-Simulated Results

This section aggregates the HPC re-simulation outputs for the candidate
allocations into a single averaged dataset.

In [ ]:
# NOTE: This code was executed on a university HPC cluster and is provided
# for documentation purposes only. File paths are environment-specific.

import pandas as pd
import os

# Step 1: Define the folder path where all CSV files are stored
folder_path = "/users/1/kuntz138/Final_Naloxone_Work"

# Step 2 & 3: Process both datasets in a loop (though only one dataset is used here)
simulation_dfs = {}  # Dictionary to store processed DataFrames (only one used)

# Create a list of file paths for 100 CSV files named naloxone_param_1.csv ... naloxone_param_100.csv
file_paths = [os.path.join(folder_path, f"naloxone_param_{i}.csv") for i in range(1, 101)]
    
# Read each CSV file into a DataFrame and store in a list
dataframes = [pd.read_csv(file) for file in file_paths]

# Concatenate all DataFrames vertically (stack rows) and reset the index
simulation_df = pd.concat(dataframes, axis=0).reset_index(drop=True)
    
# Remove the first three columns (likely metadata or unnecessary variables)
simulation_df = simulation_df.drop(simulation_df.columns[:3], axis=1)
    
# Compute the mean across simulation runs
# Assumes:
# - 100 simulations
# - Each simulation contributes 100 rows
for i in range(100):
    # Select the i-th row from each simulation block
    # (i, i+100, i+200, ..., i+9900)
    indices = [i + 100 * j for j in range(100)]
    
    # Sum the values from the first column at these indices
    aggregate_sum = simulation_df.iloc[indices, 0].sum()
    
    # Replace the i-th row's value with the average across simulations
    simulation_df.iloc[i, 0] = aggregate_sum / 100  
    
    # Keep only the first 100 rows (these now contain averaged values)
    simulation_dfs["simulation_df"] = simulation_df.iloc[:100]
    
# Extract the processed (averaged) DataFrame from the dictionary
simulation_df = simulation_dfs["simulation_df"]

In [ ]:
# Save the final DataFrame to a CSV file
output_path = os.path.join(folder_path, "best_distributions_simulation_results.csv")
simulation_df.to_csv(output_path, index=False)
print(f"Simulation results saved to: {output_path}")

## Get Results

This portion of the notebook uses the simulation results of the best distributions and produces some relevant figures for the paper.

**In plain terms:** the plots below answer two questions. First, "how does the
best-found allocation actually differ from what Rhode Island does today,
city-by-city?" (the bar chart, comparing the top 20 re-simulated candidates
against the status quo from `00_status_quo_distribution.ipynb`). Second, "how
much do the top 20 candidates actually agree with each other?" (the range
plot) — if they're all fairly close, that's a sign the optimization converged
on a genuinely good region of the allocation space, rather than landing on 20
unrelated lucky guesses.</cell id="779c0b69">

In [ ]:
from pathlib import Path

# Current notebook directory
cwd = Path.cwd()

# Navigate to repo root (go up 1 level)
repo_root = cwd.parents[0]

# Build file path
file_path_1 = repo_root / "results" / "outputs" / "best_distributions_simulation_results.csv"
file_path_2 = repo_root / "results" / "outputs" / "status_quo_distribution.csv"

In [ ]:
import pandas as pd

results_df = pd.read_csv(file_path_1)
print(results_df.shape)
status_quo_df = pd.read_csv(file_path_2, index_col=0)
status_quo_df.drop_duplicates(inplace=True)
status_quo_df.columns = status_quo_df.columns.str.replace(' ', '.', regex=False)
print(status_quo_df.shape)

In [ ]:
# Step 1: Sort by 'od_death_2026' and get the 20 smallest values
top_20_df = results_df.sort_values(by='od_death_2026').head(20)

In [ ]:
# Step 2: Prepare statistics for columns 2-40
# Exclude the first column
stats_df = top_20_df.iloc[:, 1:]  # columns 2 to 40
means = stats_df.mean()
stds = stats_df.std()

In [ ]:
# Ensure same column order as means
status_quo_values = status_quo_df.loc[0, means.index]

In [ ]:
# Step 3: Plot distribution with mean and error bars for columns 2-40
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(20, 6))

# Bars with error bars (simulation results)
plt.bar(
    means.index,
    means.values,
    yerr=stds.values,
    capsize=5,
    alpha=0.7,
    label='Simulation Mean ± Std Dev'
)

# Overlay status quo values
plt.scatter(
    means.index,
    status_quo_values.values,
    color='red',
    zorder=3,
    label='Status Quo'
)

plt.xticks(rotation=90)
plt.ylabel('Value')
plt.title('Optimized Allocations vs Status Quo')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ranges = pd.DataFrame({
    'min': top_20_df.min(),
    'max': top_20_df.max()
})

ranges = ranges.sort_values('min')

plt.figure(figsize=(10, 6))

for i, (col, row) in enumerate(ranges.iterrows()):
    plt.plot([row['min'], row['max']], [i, i], marker='o')

plt.yticks(range(len(ranges)), ranges.index)
plt.xlabel('Value')
plt.title('Range (Min to Max) of Each Column')
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
top_20_df.reset_index(drop=True)

In [ ]:
from pathlib import Path

# Current notebook directory
cwd = Path.cwd()

# Navigate to repo root (go up 1 level)
repo_root = cwd.parents[0]

# Build file path
output_dir = repo_root / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
file_path = output_dir / "metamodel_results.csv"
file_path

In [ ]:
# Export to a CSV
top_20_df.to_csv(file_path, index=False)